# Aviation Accidents Analysis

You are part of a consulting firm that is tasked to do an analysis of commercial and passenger jet airline safety. The client (an airline/airplane insurer) is interested in knowing what types of aircraft (makes/models) exhibit low rates of total destruction and low likelihood of fatal or serious passenger injuries in the event of an accident. They are also interested in any general variables/conditions that might be at play. Your analysis will be based off of aviation accident data accumulated from the years 1948-2023. 

Our client is only interested in airplane makes/models that are professional builds and could potentially still be active. Assume a max lifetime of 40 years for a make/model retirement and make sure to filter your data accordingly (i.e. from 1983 onwards). They would also like separate recommendations for small aircraft vs. larger passenger models. **In addition, make sure that claims that you make are statistically robust and that you have enough samples when making comparisons between groups.**


In this summative assessment you will demonstrate your ability to:
- **Use Pandas to load, inspect, and clean the dataset appropriately.**
- **Transform relevant columns to create measures that address the problem at hand.**
- conduct EDA: visualization and statistical measures to systematically understand the structure of the data
- recommend a set of airplanes and makes conforming to the client's request and identify at least *two* factors contributing to airplane safety. You must provide supporting evidence (visuals, summary statistics, tables) for each claim you make.

### Make relevant library imports

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

## Data Loading and Inspection

### Load in data from the relevant directory and inspect the dataframe.
- inspect NaNs, datatypes, and summary statistics

In [2]:
raw_df = pd.read_csv("data/AviationData.csv", encoding="latin1", low_memory=False)
raw_df = raw_df.rename(columns={"Aircraft.damage": "Aircraft.Damage"})

print(f"Raw shape: {raw_df.shape}")
display(raw_df.head())
display(raw_df.info())
display(raw_df.isna().mean().sort_values(ascending=False).head(15).to_frame("missing_fraction"))
display(raw_df.describe(include="all").T.head(15))


Raw shape: (88889, 31)


,Event.Id,Investigation.Type,Accident.Number,Event.Date,Location,Country,Latitude,Longitude,Airport.Code,Airport.Name,...,Purpose.of.flight,Air.carrier,Total.Fatal.Injuries,Total.Serious.Injuries,Total.Minor.Injuries,Total.Uninjured,Weather.Condition,Broad.phase.of.flight,Report.Status,Publication.Date
0,20001218X45444,Accident,SEA87LA080,1948-10-24,"MOOSE CREEK, ID",United States,NaN,NaN,NaN,NaN,...,Personal,NaN,2.0,0.0,0.0,0.0,UNK,Cruise,Probable Cause,NaN
1,20001218X45447,Accident,LAX94LA336,1962-07-19,"BRIDGEPORT, CA",United States,NaN,NaN,NaN,NaN,...,Personal,NaN,4.0,0.0,0.0,0.0,UNK,Unknown,Probable Cause,19-09-1996
2,20061025X01555,Accident,NYC07LA005,1974-08-30,"Saltville, VA",United States,36.922223,-81.878056,NaN,NaN,...,Personal,NaN,3.0,NaN,NaN,NaN,IMC,Cruise,Probable Cause,26-02-2007
3,20001218X45448,Accident,LAX96LA321,1977-06-19,"EUREKA, CA",United States,NaN,NaN,NaN,NaN,...,Personal,NaN,2.0,0.0,0.0,0.0,IMC,Cruise,Probable Cause,12-09-2000
4,20041105X01764,Accident,CHI79FA064,1979-08-02,"Canton, OH",United States,NaN,NaN,NaN,NaN,...,Personal,NaN,1.0,2.0,NaN,0.0,VMC,Approach,Probable Cause,16-04-1980


<class 'pandas.DataFrame'>
RangeIndex: 88889 entries, 0 to 88888
Data columns (total 31 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   Event.Id                88889 non-null  str    
 1   Investigation.Type      88889 non-null  str    
 2   Accident.Number         88889 non-null  str    
 3   Event.Date              88889 non-null  str    
 4   Location                88837 non-null  str    
 5   Country                 88663 non-null  str    
 6   Latitude                34382 non-null  str    
 7   Longitude               34373 non-null  str    
 8   Airport.Code            50132 non-null  str    
 9   Airport.Name            52704 non-null  str    
 10  Injury.Severity         87889 non-null  str    
 11  Aircraft.Damage         85695 non-null  str    
 12  Aircraft.Category       32287 non-null  str    
 13  Registration.Number     87507 non-null  str    
 14  Make                    88826 non-null  str    
 

None

,missing_fraction
Schedule,0.858453
Air.carrier,0.812710
FAR.Description,0.639742
Aircraft.Category,0.636772
Longitude,0.613304
Latitude,0.613203
Airport.Code,0.436016
Airport.Name,0.407081
Broad.phase.of.flight,0.305606
Publication.Date,0.154924


,count,unique,top,freq,mean,std,min,25%,50%,75%,max
Event.Id,88889,87951,20001214X45071,3,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Investigation.Type,88889,2,Accident,85015,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Accident.Number,88889,88863,ERA22LA103,2,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Event.Date,88889,14782,1982-05-16,25,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Location,88837,27758,"ANCHORAGE, AK",434,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Country,88663,219,United States,82248,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Latitude,34382,25589,332739N,19,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Longitude,34373,27154,0112457W,24,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Airport.Code,50132,10374,NONE,1488,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Airport.Name,52704,24870,Private,240,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## Data Cleaning

### Filtering aircrafts and events

We want to filter the dataset to include aircraft that the client is interested in an analysis of:
- inspect relevant columns
- figure out any reasonable imputations
- filter the dataset

In [3]:
df = raw_df.copy()
df["Event.Date"] = pd.to_datetime(df["Event.Date"], errors="coerce")

filter_steps = {
    "rows_before_filtering": len(df),
    "from_1983_onward": (df["Event.Date"].dt.year >= 1983).sum(),
    "accidents_only": (df["Investigation.Type"] == "Accident").sum(),
    "not_amateur_built": (df["Amateur.Built"] == "No").sum(),
    "airplanes_only": (df["Aircraft.Category"] == "Airplane").sum(),
}

df = df[df["Event.Date"].dt.year >= 1983].copy()
df = df[df["Investigation.Type"] == "Accident"].copy()
df = df[df["Amateur.Built"] == "No"].copy()
df = df[df["Aircraft.Category"] == "Airplane"].copy()

# This keeps the sample aligned to the client request while avoiding
# ambiguous aircraft-category imputations for older records.
print("Filtering checkpoints:")
display(pd.Series(filter_steps, name="row_count").to_frame())
print(f"Shape after filtering: {df.shape}")
display(df[[
    "Event.Date",
    "Investigation.Type",
    "Aircraft.Category",
    "Amateur.Built",
    "Make",
    "Model",
]].head())


Filtering checkpoints:


,row_count
rows_before_filtering,88889
from_1983_onward,85289
accidents_only,85015
not_amateur_built,80312
airplanes_only,27617


Shape after filtering: (19937, 31)


,Event.Date,Investigation.Type,Aircraft.Category,Amateur.Built,Make,Model
4171,1983-03-20,Accident,Airplane,No,Piper,PA-28-140
4285,1983-04-02,Accident,Airplane,No,De Havilland,DHC-6
5960,1983-08-21,Accident,Airplane,No,Lockheed,"LEARSTAR, L-18-56"
6669,1983-10-28,Accident,Airplane,No,Short Brothers,SD3-30
6806,1983-11-13,Accident,Airplane,No,Beech,C35


### Cleaning and constructing Key Measurables

Injuries and robustness to destruction are a key interest point for the client. Clean and impute relevant columns and then create derived fields that best quantifies what the client wishes to track. **Use commenting or markdown to explain any cleaning assumptions as well as any derived columns you create.**

**Construct metric for fatal/serious injuries**

*Hint:* Estimate the total number of passengers on each flight. The likelihood of serious / fatal injury can be estimated as a fraction from this.

In [4]:
injury_cols = [
    "Total.Fatal.Injuries",
    "Total.Serious.Injuries",
    "Total.Minor.Injuries",
    "Total.Uninjured",
]

for col in injury_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce").fillna(0)

# Approximate total passengers/occupants as the total across injury outcomes.
# Rows with zero observed occupants are treated as unknown for rate calculations.
df["Total.Passengers"] = df[injury_cols].sum(axis=1)
df.loc[df["Total.Passengers"] == 0, "Total.Passengers"] = np.nan

df["Fatal.Serious.Count"] = (
    df["Total.Fatal.Injuries"] + df["Total.Serious.Injuries"]
)
df["Fatal.Serious.Fraction"] = (
    df["Fatal.Serious.Count"] / df["Total.Passengers"]
)

display(
    df[[
        "Total.Passengers",
        "Fatal.Serious.Count",
        "Fatal.Serious.Fraction",
    ]].describe()
)
print(
    "Rows with usable fatal/serious fraction:",
    df["Fatal.Serious.Fraction"].notna().sum(),
)


,Total.Passengers,Fatal.Serious.Count,Fatal.Serious.Fraction
count,19674.000000,19937.000000,19674.000000
mean,6.187303,0.983498,0.296024
std,28.672968,7.030694,0.436755
min,1.000000,0.000000,0.000000
25%,1.000000,0.000000,0.000000
50%,2.000000,0.000000,0.000000
75%,2.000000,1.000000,1.000000
max,576.000000,295.000000,1.000000


Rows with usable fatal/serious fraction: 19674


**Aircraft.Damage**
- identify and execute any cleaning tasks
- construct a derived column tracking whether an aircraft was destroyed or not.

In [5]:
damage_clean = (
    df["Aircraft.Damage"]
    .astype("string")
    .str.strip()
    .str.title()
    .replace({"Unknown": pd.NA})
)

df["Aircraft.Damage.Clean"] = damage_clean
df["Destroyed"] = (
    df["Aircraft.Damage.Clean"].eq("Destroyed").fillna(False).astype(int)
)

display(df["Aircraft.Damage.Clean"].value_counts(dropna=False).to_frame("count"))
display(df["Destroyed"].value_counts().rename(index={0: "Not Destroyed", 1: "Destroyed"}).to_frame("count"))


,count
Aircraft.Damage.Clean,
Substantial,16965
Destroyed,2316
<NA>,509
Minor,147


,count
Destroyed,
Not Destroyed,17621
Destroyed,2316


### Investigate the *Make* column
- Identify cleaning tasks here
- List cleaning tasks clearly in markdown
- Execute the cleaning tasks
- For your analysis, keep Makes with a reasonable number (you can put the threshold at 50 though lower could work as well)

In [6]:
make_aliases = {
    "AVIAT AIRCRAFT INC": "AVIAT",
    "AVIAT AIRCRAFT": "AVIAT",
    "AVIAT INC": "AVIAT",
    "MCDONNELL DOUGLAS AIRCRAFT CO": "MCDONNELL DOUGLAS",
    "MCDONNELL DOUGLAS CORPORATION": "MCDONNELL DOUGLAS",
    "THE BOEING COMPANY": "BOEING",
    "BOEING COMPANY": "BOEING",
    "CESSNA AIRCRAFT CO": "CESSNA",
    "PIPER AIRCRAFT INC": "PIPER",
    "AIR TRACTOR INC": "AIR TRACTOR",
    "DEHAVILLAND": "DE HAVILLAND",
    "DEHAVILLAND CANADA": "DE HAVILLAND",
    "DIAMOND AIRCRAFT IND INC": "DIAMOND AIRCRAFT",
    "DIAMOND AIRCRAFT INDUSTRIES": "DIAMOND AIRCRAFT",
    "DIAMOND AIRCRAFT IND GMBH": "DIAMOND AIRCRAFT",
    "DIAMOND AIRCRAFT": "DIAMOND AIRCRAFT",
    "DIAMOND AIRCRAFT INDUSTRIES IN": "DIAMOND AIRCRAFT",
    "DIAMOND AICRAFT INDUSTRIES INC": "DIAMOND AIRCRAFT",
    "RAYTHEON AIRCRAFT COMPANY": "RAYTHEON",
    "RAYTHEON CORPORATE JETS": "RAYTHEON",
    "AIRBUS INDUSTRIE": "AIRBUS",
    "EMBRAER S A": "EMBRAER",
    "EMBRAER EMPRESA BRASILEIRA DE": "EMBRAER",
    "EMBRAER SA": "EMBRAER",
    "EMBRAER AIRCRAFT": "EMBRAER",
    "GRUMMAN ACFT ENG COR SCHWEIZER": "GRUMMAN SCHWEIZER",
    "SCHWEIZER AIRCRAFT CORP": "GRUMMAN SCHWEIZER",
    "GRUMMAN AMERICAN AVN CORP": "GRUMMAN AMERICAN",
    "GRUMMAN AMERICAN AVIATION CORP": "GRUMMAN AMERICAN",
    "AMERICAN CHAMPION AIRCRAFT": "CHAMPION",
    "ROCKWELL INTERNATIONAL": "ROCKWELL",
    "AYRES CORPORATION": "AYRES",
}

df["Make.Clean"] = (
    df["Make"]
    .astype("string")
    .str.upper()
    .str.replace(r"[^A-Z0-9]+", " ", regex=True)
    .str.strip()
    .replace({"": pd.NA})
    .replace(make_aliases)
)

make_counts = df["Make.Clean"].value_counts()

display(make_counts.head(25).to_frame("accident_count"))

# Keep makes with enough observations for stable comparisons.
df = df[df["Make.Clean"].map(make_counts) >= 50].copy()
print(f"Shape after make-count threshold: {df.shape}")


,accident_count
Make.Clean,
CESSNA,7053
PIPER,3968
BEECH,1392
BOEING,557
AIR TRACTOR,430
MOONEY,358
CIRRUS DESIGN CORP,234
BELLANCA,217
MAULE,215


Shape after make-count threshold: (16944, 37)


### Inspect Model column
- Get rid of any NaNs.
- Inspect the column and counts for each model/make. Are model labels unique to each make?
- If not, create a derived column that is a unique identifier for a given plane type.

In [7]:
df["Model.Clean"] = (
    df["Model"]
    .astype("string")
    .str.upper()
    .str.replace(r"[^A-Z0-9]+", " ", regex=True)
    .str.strip()
    .replace({"": pd.NA})
)

df = df.dropna(subset=["Make.Clean", "Model.Clean"]).copy()
df["Plane.Type"] = df["Make.Clean"] + " " + df["Model.Clean"]

shared_model_names = (
    df.groupby("Model.Clean")["Make.Clean"]
    .nunique()
    .sort_values(ascending=False)
)

# Plane.Type is the make + model identifier used in analysis because
# many model labels recur across different manufacturers.
plane_type_passenger_profile = (
    df.groupby("Plane.Type")["Total.Passengers"].median()
)
df["Plane.Size"] = np.where(
    df["Plane.Type"].map(plane_type_passenger_profile) >= 20,
    "Large",
    "Small",
)

print("Models shared across multiple makes:", (shared_model_names > 1).sum())
display(shared_model_names.head(15).to_frame("distinct_makes"))
display(df["Plane.Size"].value_counts().to_frame("count"))


Models shared across multiple makes: 65


,distinct_makes
Model.Clean,
S2R,3
500,3
G 164,3
G 164A,3
7BCM,2
B36TC,2
S2R R1340,2
7EC,2
SF50,2


,count
Plane.Size,
Small,16324
Large,609


### Cleaning other columns
- there are other columns containing data that might be related to the outcome of an accident. We list a few here:
- Engine.Type
- Weather.Condition
- Number.of.Engines
- Purpose.of.flight
- Broad.phase.of.flight

Inspect and identify potential cleaning tasks in each of the above columns. Execute those cleaning tasks. 

**Note**: You do not necessarily need to impute or drop NaNs here.

In [8]:
df["Number.of.Engines"] = pd.to_numeric(df["Number.of.Engines"], errors="coerce")

df["Engine.Type.Clean"] = (
    df["Engine.Type"]
    .astype("string")
    .str.upper()
    .str.replace(r"[^A-Z0-9]+", " ", regex=True)
    .str.strip()
    .replace({"": pd.NA, "UNKNOWN": pd.NA, "UNK": pd.NA, "NONE": pd.NA})
)

df["Weather.Condition.Clean"] = (
    df["Weather.Condition"]
    .astype("string")
    .str.upper()
    .str.strip()
    .replace({"": pd.NA, "UNKNOWN": pd.NA, "UNK": pd.NA})
)

df["Purpose.of.Flight.Clean"] = (
    df["Purpose.of.flight"]
    .astype("string")
    .str.upper()
    .str.replace(r"[^A-Z0-9]+", " ", regex=True)
    .str.strip()
    .replace({"": pd.NA, "UNKNOWN": pd.NA})
)

df["Broad.Phase.Clean"] = (
    df["Broad.phase.of.flight"]
    .astype("string")
    .str.upper()
    .str.replace(r"[^A-Z0-9]+", " ", regex=True)
    .str.strip()
    .replace({"": pd.NA, "UNKNOWN": pd.NA})
)

for col in [
    "Engine.Type.Clean",
    "Weather.Condition.Clean",
    "Purpose.of.Flight.Clean",
    "Broad.Phase.Clean",
]:
    print(f"\n{col}")
    display(df[col].value_counts(dropna=False).head(12).to_frame("count"))



Engine.Type.Clean


,count
Engine.Type.Clean,
RECIPROCATING,12939
<NA>,2581
TURBO PROP,953
TURBO FAN,404
TURBO JET,47
TURBO SHAFT,9



Weather.Condition.Clean


,count
Weather.Condition.Clean,
VMC,14306
<NA>,1731
IMC,896



Purpose.of.Flight.Clean


,count
Purpose.of.Flight.Clean,
PERSONAL,9934
INSTRUCTIONAL,2423
<NA>,2229
AERIAL APPLICATION,818
BUSINESS,402
POSITIONING,260
SKYDIVING,156
AERIAL OBSERVATION,146
OTHER WORK USE,120



Broad.Phase.Clean


,count
Broad.Phase.Clean,
<NA>,14460
LANDING,1127
TAKEOFF,432
CRUISE,237
APPROACH,212
MANEUVERING,140
TAXI,94
GO AROUND,84
DESCENT,61


### Column Removal
- inspect the dataframe and drop any columns that have too many NaNs

In [9]:
keep_even_if_sparse = {
    "Broad.Phase.Clean",
    "Weather.Condition.Clean",
    "Engine.Type.Clean",
    "Purpose.of.Flight.Clean",
    "Number.of.Engines",
}

missing_fraction = df.isna().mean().sort_values(ascending=False)
drop_candidates = [
    col
    for col, frac in missing_fraction.items()
    if frac > 0.50 and col not in keep_even_if_sparse
]

# Drop highly incomplete columns that are not part of the downstream analysis.
drop_candidates.extend([
    "Schedule",
    "Air.carrier",
    "Airport.Code",
    "Airport.Name",
    "Registration.Number",
    "Latitude",
    "Longitude",
    "Publication.Date",
    "Report.Status",
    "Make",
    "Model",
    "Engine.Type",
    "Weather.Condition",
    "Purpose.of.flight",
    "Broad.phase.of.flight",
    "Aircraft.Damage",
])

drop_candidates = sorted(set(col for col in drop_candidates if col in df.columns))
df = df.drop(columns=drop_candidates)

print("Dropped columns:")
print(drop_candidates)
print(f"Final cleaned shape: {df.shape}")
display(df.head())


Dropped columns:
['Air.carrier', 'Aircraft.Damage', 'Airport.Code', 'Airport.Name', 'Broad.phase.of.flight', 'Engine.Type', 'Latitude', 'Longitude', 'Make', 'Model', 'Publication.Date', 'Purpose.of.flight', 'Registration.Number', 'Report.Status', 'Schedule', 'Weather.Condition']
Final cleaned shape: (16933, 28)


,Event.Id,Investigation.Type,Accident.Number,Event.Date,Location,Country,Injury.Severity,Aircraft.Category,Amateur.Built,Number.of.Engines,...,Aircraft.Damage.Clean,Destroyed,Make.Clean,Model.Clean,Plane.Type,Plane.Size,Engine.Type.Clean,Weather.Condition.Clean,Purpose.of.Flight.Clean,Broad.Phase.Clean
4171,20001214X42331,Accident,ATL83FA140,1983-03-20,"CROSSVILLE, TN",United States,Fatal(1),Airplane,No,1.0,...,Destroyed,1,PIPER,PA 28 140,PIPER PA 28 140,Small,RECIPROCATING,IMC,PERSONAL,CRUISE
4285,20001214X42672,Accident,FTW83LA177,1983-04-02,"MCKINNEY, TX",United States,Fatal(1),Airplane,No,2.0,...,<NA>,0,DE HAVILLAND,DHC 6,DE HAVILLAND DHC 6,Small,TURBO PROP,VMC,SKYDIVING,STANDING
6806,20001214X45188,Accident,NYC84LA028,1983-11-13,"MARTHA'S VINEYARD, MA",United States,Non-Fatal,Airplane,No,1.0,...,Substantial,0,BEECH,C35,BEECH C35,Small,RECIPROCATING,VMC,PERSONAL,CLIMB
7084,20001214X45339,Accident,LAX84LA110,1983-12-22,"SANTA ROSA ISLAND, CA",United States,Non-Fatal,Airplane,No,1.0,...,Substantial,0,CESSNA,180K,CESSNA 180K,Small,RECIPROCATING,VMC,PERSONAL,TAKEOFF
7708,20001214X38957,Accident,ATL84LA120,1984-03-14,"MYRTLE BEACH, SC",United States,Non-Fatal,Airplane,No,2.0,...,Substantial,0,BEECH,99,BEECH 99,Small,TURBO PROP,VMC,<NA>,LANDING


### Save DataFrame to csv
- its generally useful to save data to file/server after its in a sufficiently cleaned or intermediate state
- the data can then be loaded directly in another notebook for further analysis
- this helps keep your notebooks and workflow readable, clean and modularized

In [10]:
output_path = "data/aviation_accidents_cleaned.csv"
df.to_csv(output_path, index=False)

print(f"Saved cleaned dataset to: {output_path}")


Saved cleaned dataset to: data/aviation_accidents_cleaned.csv
